In [1]:
import numpy as np
import torch
import json
from model import Transformer
from data import Dataset_Basic, DataLoader
from utils import append_positional_encoding, identity_pe, get_pe, get_accuracy, get_close_accuracy

/u4/ggiapitz/.conda/envs/pytorch_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
params_file = 'experiment_params.json'

with open(params_file, 'r') as fp:
    data = json.load(fp)

In [15]:
def run_inference(target, data, device, model_path, runs = 10):
    """Run experiment with given parameters and return final losses for standard and positional transformer

    Args:
        target (str): target function (one of 'sum', 'min', 'median', 'sort', 'minsum')
        data (dict): dictionary containing parameters for the experiment
        device (torch.device): device to run the experiment on
        experiment (str, optional): type of experiment (one of 'sample_complexity', 'scale_generalization`,
        `size`). Defaults to 'sample_complexity'.
        i (int, optional): If sample_complexity is True, this is the index of data['num_train_samples']. Otherwise
        it the index of data['low_test'] and data['high_test']. Defaults to 0.
    """
    n = data['n'][0]
    num_test_samples = data['num_test_samples']
    num_additional_node = data['num_additional_node']
    batch_size = data['batch_size']
    low_train = data['low_train']
    high_train = data['high_train']
    cumulative = data['cumulative']
    use_integer = True
    variable_length = data['variable_length'] if 'variable_length' in data else False

    if target == 'minsum':
        pos_enc_base = identity_pe(2*n+num_additional_node).to(device)
    else:
        pos_enc_base = identity_pe(n+num_additional_node).to(device)

    if target == 'path':
        data_dim = n
    else:
        data_dim = 1

    in_dim_s = data_dim + pos_enc_base.size(1)
    in_dim_p = data_dim
    out_dim = data_dim
    embed_dim = data['embed_dim']
    num_heads = data['num_heads']
    use_rope = data['RoPE'] if 'RoPE' in data else False
    num_layers = np.log2(n).astype(int) + 1 if 'model_num_layers' not in data else data['model_num_layers']
    mlp_hidden_dim = data['mlp_hidden_dim']
    mlp_num_layers = data['mlp_num_layers']

    test_datasets = [Dataset_Basic(num_samples=num_test_samples, length=n, low=low_test, high=high_test, target=target, use_integer=use_integer, cumulative=cumulative, num_additional_node=num_additional_node, reject_low=low_train, reject_high=high_train, variable_length=variable_length) for low_test, high_test in zip(data['low_test'], data['high_test'])]
    test_loaders = [DataLoader(test_dataset, batch_size=batch_size, shuffle=False, variable_length=variable_length) for test_dataset in test_datasets]

    models_s = [Transformer(in_dim=in_dim_s, embed_dim=embed_dim, out_dim=out_dim, num_heads=num_heads, num_layers=num_layers,
                        mlp_hidden_dim=mlp_hidden_dim, mlp_num_layers=mlp_num_layers, positional=False, RoPE=use_rope, pos_dim=pos_enc_base.size(1)).to(device) for _ in range(runs)]
    models_p = [Transformer(in_dim=in_dim_p, embed_dim=embed_dim, out_dim=out_dim, num_heads=num_heads, num_layers=num_layers,
                            mlp_hidden_dim=mlp_hidden_dim, mlp_num_layers=mlp_num_layers, positional=True, pos_dim=pos_enc_base.size(1)).to(device) for _ in range(runs)]

    for i in range(runs):   
        models_s[i].load_state_dict(torch.load(model_path + f"/run{i+1}_standard.pt", map_location=device))
        models_p[i].load_state_dict(torch.load(model_path + f"/run{i+1}_positional.pt", map_location=device))

    test_acc_s = [0] * len(test_loaders)
    test_acc_p = [0] * len(test_loaders)
    for i, test_loader in enumerate(test_loaders):
        for (x, y) in test_loader:
            for j in range(runs):
                x, y = x.to(device), y.to(device)
                pos_enc = get_pe(pos_enc_base, x, num_additional_node) if variable_length else pos_enc_base
                x_app = append_positional_encoding(x, pos_enc)
                out = models_s[j](x_app, p=pos_enc)
                rounded_out = torch.round(out)
                test_acc_s[i] += get_close_accuracy(out, y, num_additional_node, n, target, rtol=0.1, atol=0.1)
                out = models_p[j](x, p=pos_enc)
                rounded_out = torch.round(out)
                test_acc_p[i] += get_close_accuracy(out, y, num_additional_node, n, target, rtol=0.1, atol=0.1)
            
            test_acc_s[i] /= runs
            test_acc_p[i] /= runs

    print(test_acc_s)
    print(test_acc_p)

In [16]:
run_inference('median', data, device, 'median_scale/models')

/tmp/ipykernel_1038181/2644090848.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  models_s[i].load_state_dict(torch.load(model_path + f"/run{i+1}_standard.pt", map_loca

[0.8388, 0.0683, 0.0048000000000000004, 0.001, 0.0012000000000000001, 0.0001, 0.0004, 0.0001, 0.0, 0.00030000000000000003]
[0.9762000000000001, 0.8628, 0.7763, 0.6943, 0.6375, 0.5669000000000001, 0.557, 0.5264, 0.4732, 0.4567]
